# Pre-Training Configuration

## Purpose
This notebook is run **once by the trainer** before the training starts.  
It creates isolated environments for each participant:

1. **Catalog** per user: `retailhub_{username}`
2. **Schemas**: `bronze`, `silver`, `gold`, `default`
3. **Volume**: `datasets` (managed) in the `default` schema
4. **Dataset files**: copied from the Git repo to each user's Volume
5. **Permissions**: full access granted to each participant on their catalog
6. **Workspace access**: the training group is assigned to the workspace and has `workspace-access` (Step 4b)

**After running this notebook**, participants can run `00_setup.ipynb` to validate their environment.

### Checklist before running:
- [ ] Databricks workspace is ready
- [ ] Training group exists with all participants added
- [ ] This repo is cloned as a Git folder in the workspace
- [ ] You have `CREATE CATALOG` and `MANAGE` privileges (account admin or metastore admin)
- [ ] Storage location for managed catalogs is configured
- [ ] Workspace admin (identity-federated workspace) or account admin — needed for Step 5 (account group `analysts` for Day 3 governance)

## Step 0: Configuration

Adjust the values below for your training session.

In [ ]:
# =============================================================================
# CONFIGURATION -- Adjust these values for your training
# =============================================================================

# Databricks group containing all training participants
TRAINING_GROUP = "alt_trn_gr"

# Catalog naming: retailhub_{username}
CATALOG_PREFIX = "retailhub"

# Managed location for catalogs (ADLS, S3, or GCS path)
# Example Azure: "abfss://container@account.dfs.core.windows.net/path"
# Example AWS:   "s3://bucket/path"
# Leave empty to use the metastore default location
# Terraform output external_location_url (UC external location el-dea160926-training-external).
# No trailing slash - catalogs land in <STORAGE_LOCATION>/<catalog_name>.
STORAGE_LOCATION = "abfss://external@stdea160926trainingagbr2.dfs.core.windows.net"

# Schema names (Medallion architecture)
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DEFAULT_SCHEMA = "default"

# Volume name for datasets
VOLUME_NAME = "datasets"

# Trainer account mapping: login prefix -> catalog slug; edit per delivery.
# Any user whose login contains a key below gets catalog retailhub_<slug>.
TRAINER_OVERRIDES = {"krzysztof.burejza": "trainer", "trener02": "trainer"}

print(f"Training group:   {TRAINING_GROUP}")
print(f"Catalog prefix:   {CATALOG_PREFIX}")
print(f"Storage location: {STORAGE_LOCATION or '(metastore default)'}")

## Step 1: Get Participants from Training Group

Uses the Databricks SCIM API to retrieve group members and generate catalog names.

In [ ]:
import requests
import re

def get_group_members(group_name):
    """
    Get all members of a Databricks group using the SCIM REST API.
    Returns list of usernames (email addresses).
    """
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host = context.apiUrl().get()
    token = context.apiToken().get()

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    # Find the group by name. The workspace SCIM view only lists groups that are
    # ASSIGNED to this workspace; fall back to the account SCIM API (workspace
    # proxy, identity-federated workspaces) so an account group that is not yet
    # assigned is still found.
    params = {"filter": f'displayName eq "{group_name}"'}
    for api in ["preview/scim/v2", "account/scim/v2"]:
        response = requests.get(f"{host}/api/2.0/{api}/Groups", headers=headers, params=params)
        if response.ok and response.json().get("Resources"):
            break
    else:
        raise ValueError(f"Group '{group_name}' not found (workspace or account)")
    if api == "account/scim/v2":
        print(f"NOTE: '{group_name}' found at ACCOUNT level only -- assign it to this workspace "
              f"(Account console > Workspaces > Permissions) so participants can log in.")

    group_id = response.json()["Resources"][0]["id"]

    # Get group details with members
    response = requests.get(f"{host}/api/2.0/{api}/Groups/{group_id}", headers=headers)
    response.raise_for_status()

    members = response.json().get("members", [])

    # Resolve each member to an email (nested groups are skipped)
    user_emails = []
    for member in members:
        if member.get("$ref", "Users/").startswith("Users/"):
            user_response = requests.get(f"{host}/api/2.0/{api}/Users/{member['value']}", headers=headers)
            if user_response.status_code == 404:
                continue  # not a user (nested group)
            user_response.raise_for_status()
            user = user_response.json()
            emails = user.get("emails", [])
            email = emails[0].get("value", "") if emails else user.get("userName", "")
            if email:
                user_emails.append(email)

    return user_emails


def sanitize_username(email):
    """
    Convert email to a safe catalog name suffix.
    Example: jan.kowalski@company.com -> jan_kowalski
    Example: student501@gmail.com -> student501
    """
    username = email.split('@')[0]
    safe_name = re.sub(r'[^a-zA-Z0-9]', '_', username).lower()
    safe_name = re.sub(r'_+', '_', safe_name)
    safe_name = re.sub(r'^[0-9_]+', '', safe_name)  # catalog cant start with digit
    safe_name = safe_name.strip('_')
    return safe_name or "user"


print("Functions defined: get_group_members(), sanitize_username()")

In [ ]:
# =============================================================================
# Get users and build catalog mapping
# =============================================================================
try:
    training_users = get_group_members(TRAINING_GROUP)
    print(f"Found {len(training_users)} users in group '{TRAINING_GROUP}':")
    print("=" * 60)

    user_catalog_map = {}
    for email in sorted(training_users):
        safe_name = sanitize_username(email)

        # Map trainer accounts (TRAINER_OVERRIDES, Step 0) to a dedicated slug
        override = next(
            (slug for prefix, slug in TRAINER_OVERRIDES.items()
             if prefix in email.lower().split('@')[0]),
            None,
        )
        catalog_name = f"{CATALOG_PREFIX}_{override or safe_name}"

        user_catalog_map[email] = catalog_name
        print(f"  {email:<40} -> {catalog_name}")

    print("=" * 60)
    print(f"Total: {len(set(user_catalog_map.values()))} unique catalogs to create")

except Exception as e:
    print(f"ERROR: {e}")
    print(f"\nPossible issues:")
    print(f"  1. Group '{TRAINING_GROUP}' does not exist")
    print(f"  2. You don't have permission to read group members")
    print(f"  3. Group has no members")
    raise

## Step 2: Create Catalogs, Schemas, and Volumes

For each participant:
- Catalog: `retailhub_{username}`
- Schemas: `bronze`, `silver`, `gold`, `default`
- Volume: `datasets` in `default` schema
- Permissions: `ALL PRIVILEGES` granted to the user and the trainer

In [0]:
def create_user_environment(email, catalog_name, storage_location):
    """
    Create catalog, schemas, volume and set permissions for a training participant.
    """
    results = {"catalog": False, "schemas": [], "volume": False, "permissions": False}
    trainer_email = spark.sql("SELECT current_user()").first()[0]

    try:
        # Create catalog (with or without managed location)
        if storage_location:
            spark.sql(f"""
                CREATE CATALOG IF NOT EXISTS {catalog_name}
                MANAGED LOCATION '{storage_location}/{catalog_name}'
            """)
        else:
            spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
        results["catalog"] = True

        # Create Medallion schemas + default
        for schema in [DEFAULT_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
            spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema}")
            results["schemas"].append(schema)

        # Create managed Volume for datasets
        spark.sql(f"""
            CREATE VOLUME IF NOT EXISTS {catalog_name}.{DEFAULT_SCHEMA}.{VOLUME_NAME}
            COMMENT 'Training datasets for RetailHub project'
        """)
        results["volume"] = True

        # Grant permissions
        spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG {catalog_name} TO `{email}`")
        if trainer_email != email:
            spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG {catalog_name} TO `{trainer_email}`")
        results["permissions"] = True

    except Exception as e:
        results["error"] = str(e)

    return results

In [0]:
# =============================================================================
# Create environments for all participants
# =============================================================================
creation_results = {}

for email, catalog_name in user_catalog_map.items():
    print(f"Processing: {email}")
    result = create_user_environment(email, catalog_name, STORAGE_LOCATION)
    creation_results[email] = result

    if "error" in result:
        print(f"  ERROR: {result['error']}")
    else:
        print(f"  Catalog:     {catalog_name}")
        print(f"  Schemas:     {', '.join(result['schemas'])}")
        print(f"  Volume:      {result['volume']}")
        print(f"  Permissions: {result['permissions']}")
    print()

successful = sum(1 for r in creation_results.values() if "error" not in r)
print(f"Result: {successful}/{len(creation_results)} environments created successfully")

## Step 3: Copy Dataset Files to Volumes

Copies the entire `dataset/` folder from the Git repo to each user's Volume.  
Uses `shutil.copytree` for a complete recursive copy.

**Dataset structure in each Volume:**
```
/Volumes/retailhub_{user}/default/datasets/
  customers/
    customers.csv
    customers_extended.csv
    customers_new.csv
  orders/
    orders_batch.json
    stream/
      orders_stream_001.json ... 003.json
  products/
    products.csv
  demo/ingestion/
    orders/stream/
      orders_stream_004.json ... 006.json
  workshop/
    Customers.csv, Product.csv, ...
    Lakeflow/
      Customers/Customers.csv, ...
```

In [0]:
import shutil
import os

def copy_dataset_to_volume(catalog_name):
    """
    Copy dataset files from the Git repo to a user's Volume.
    Skips .DS_Store and other hidden files.
    """
    # Source: dataset/ folder in the repo root
    # When running from notebooks/setup/, repo root is ../../
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    source_path = os.path.join(repo_root, "dataset")

    # Target: Volume path
    volume_path = f"/Volumes/{catalog_name}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"

    if not os.path.exists(source_path):
        return {"error": f"Source not found: {source_path}"}

    try:
        # Copy with ignore for hidden files
        def ignore_hidden(directory, files):
            return [f for f in files if f.startswith('.')]

        shutil.copytree(source_path, volume_path, dirs_exist_ok=True, ignore=ignore_hidden)
        return {"success": True}
    except Exception as e:
        return {"error": str(e)}

In [0]:
# =============================================================================
# Copy dataset to each participant's Volume
# =============================================================================
print("Copying dataset to user Volumes...")
print("=" * 60)

copy_results = {}

for email, catalog_name in user_catalog_map.items():
    print(f"  {catalog_name}... ", end="")
    result = copy_dataset_to_volume(catalog_name)
    copy_results[email] = result

    if "error" in result:
        print(f"FAILED: {result['error']}")
    else:
        print("OK")

successful = sum(1 for r in copy_results.values() if r.get("success"))
print("=" * 60)
print(f"Result: {successful}/{len(copy_results)} volumes populated")

## Step 4: Verification

Final check that all environments are ready for training.

In [0]:
# =============================================================================
# VERIFICATION -- Check all environments
# =============================================================================
print("=" * 60)
print("TRAINING ENVIRONMENT SUMMARY")
print("=" * 60)
print()

all_ok = True

for email, catalog_name in user_catalog_map.items():
    status = []

    # Check catalog
    try:
        spark.sql(f"USE CATALOG {catalog_name}")
        status.append("catalog: OK")
    except:
        status.append("catalog: MISSING")
        all_ok = False

    # Check schemas
    try:
        schemas = [row[0] for row in spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()]
        missing = [s for s in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA] if s not in schemas]
        if missing:
            status.append(f"schemas: MISSING {missing}")
            all_ok = False
        else:
            status.append("schemas: OK")
    except:
        status.append("schemas: ERROR")
        all_ok = False

    # Check Volume
    volume_path = f"/Volumes/{catalog_name}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"
    try:
        files = dbutils.fs.ls(volume_path)
        file_count = len(files)
        status.append(f"volume: OK ({file_count} items)")
    except:
        status.append("volume: MISSING")
        all_ok = False

    print(f"  {email}")
    print(f"    Catalog: {catalog_name}")
    print(f"    Status:  {' | '.join(status)}")
    print()

print("=" * 60)
if all_ok:
    print("ALL ENVIRONMENTS READY -- Training can begin!")
    print("Participants should run: notebooks/setup/00_setup.ipynb")
else:
    print("WARNING: Some environments have issues. Check the details above.")
print("=" * 60)

## Step 4b: Workspace access for participants (do not skip!)

A participant can have a catalog and still be **unable to open the workspace**. Two things are required:

1. The training group (`TRAINING_GROUP`, an **account** group) is **assigned to this workspace** with permission *User*
   (Account console → **Workspaces** → this workspace → **Permissions** → *Add permissions*).
2. The group has the **entitlements** `workspace-access` (open the workspace UI) and `databricks-sql-access` (SQL editor / warehouses).
   On this training workspace the built-in `users` group had **no** entitlements, so assignment alone was not enough.

The cell below checks both, adds the missing entitlements (workspace admin required) and prints what is still manual.

In [ ]:
# =============================================================================
# STEP 4b -- make sure every participant can actually log in (idempotent)
# =============================================================================
import requests

REQUIRED_ENTITLEMENTS = ["workspace-access", "databricks-sql-access"]

_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_host = _ctx.apiUrl().get()
_hdr = {"Authorization": f"Bearer {_ctx.apiToken().get()}", "Content-Type": "application/scim+json"}

def _ws_group(name):
    r = requests.get(f"{_host}/api/2.0/preview/scim/v2/Groups", headers=_hdr,
                     params={"filter": f'displayName eq "{name}"'})
    r.raise_for_status()
    res = r.json().get("Resources", [])
    return requests.get(f"{_host}/api/2.0/preview/scim/v2/Groups/{res[0]['id']}", headers=_hdr).json() if res else None

grp = _ws_group(TRAINING_GROUP)
if grp is None:
    print(f"[MANUAL] '{TRAINING_GROUP}' is NOT assigned to this workspace -> participants cannot log in.")
    print("         Account console -> Workspaces -> <this workspace> -> Permissions -> Add -> "
          f"'{TRAINING_GROUP}' with permission 'User', then re-run this cell.")
else:
    have = {e["value"] for e in grp.get("entitlements", [])}
    missing = [e for e in REQUIRED_ENTITLEMENTS if e not in have]
    if missing:
        r = requests.patch(
            f"{_host}/api/2.0/preview/scim/v2/Groups/{grp['id']}", headers=_hdr,
            json={"schemas": ["urn:ietf:params:scim:api:messages:2.0:PatchOp"],
                  "Operations": [{"op": "add", "path": "entitlements",
                                  "value": [{"value": e} for e in missing]}]})
        if r.ok:
            print(f"[ADDED]  entitlements {missing} on group '{TRAINING_GROUP}'")
        else:
            print(f"[MANUAL] could not add {missing} ({r.status_code}): {r.text[:200]}")
            print("         Settings -> Identity and access -> Groups -> "
                  f"'{TRAINING_GROUP}' -> Entitlements -> tick Workspace access + Databricks SQL access")
    else:
        print(f"[OK]     group '{TRAINING_GROUP}' has {REQUIRED_ENTITLEMENTS}")

    grp = _ws_group(TRAINING_GROUP)
    members = [m.get("display") for m in grp.get("members", [])]
    print(f"[OK]     '{TRAINING_GROUP}' is assigned to this workspace — {len(members)} member(s): {members}")
    print("         Ask one participant to open the workspace URL before Day 1 (the only end-to-end proof).")

## Step 5: Governance demo group `analysts` (Day 3 — module 10 / lab 10)

Module 10 and lab 10 grant Unity Catalog privileges to the group **`analysts`**.

> ⚠️ **Must be an ACCOUNT group.** Unity Catalog privileges can only be granted to account-level identities — **workspace-local groups cannot be granted access to Unity Catalog data**. That is why this step does **not** use `WorkspaceClient().groups.create()` (the workspace SCIM API creates *workspace-local* groups); it calls the **Account Groups API through the workspace proxy** (`/api/2.0/account/scim/v2/Groups`), which workspace admins of identity-federated workspaces may use.

If the call is not permitted, create the group manually (it can stay empty — participants only need the GRANT statements to succeed):
- Account console → **User management** → **Groups** → **Add group** → `analysts`, or
- Workspace → **Settings → Identity and access → Groups → Manage → Add Group → Add new** → `analysts`

Without the group, lab 10 Tasks 1 and 9 and the GRANT cells in module 10 print `[SKIPPED]` instead of failing.


In [ ]:
# =============================================================================
# STEP 5 -- ensure the ACCOUNT group(s) used in Day 3 governance exist (idempotent)
# =============================================================================
GOVERNANCE_GROUPS = ["analysts"]

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = context.apiUrl().get()
headers = {"Authorization": f"Bearer {context.apiToken().get()}", "Content-Type": "application/json"}

def find_group(api, name):
    """api: 'account/scim/v2' (account groups via workspace proxy) or 'preview/scim/v2' (workspace view)."""
    r = requests.get(f"{host}/api/2.0/{api}/Groups", headers=headers,
                     params={"filter": f'displayName eq "{name}"'})
    r.raise_for_status()
    return r.json().get("Resources", [])

for name in GOVERNANCE_GROUPS:
    try:
        # A workspace-local group with the same name blocks the account group -> warn
        for g in find_group("preview/scim/v2", name):
            if g.get("meta", {}).get("resourceType") == "WorkspaceGroup":
                print(f"  WARNING: '{name}' exists as a WORKSPACE-LOCAL group -- UC grants to it fail. "
                      f"Rename it (e.g. '{name} (workspace)') and re-run this cell.")
        if find_group("account/scim/v2", name):
            print(f"  [OK]      account group '{name}' already exists")
            continue
        r = requests.post(f"{host}/api/2.0/account/scim/v2/Groups", headers=headers, json={"displayName": name})
        r.raise_for_status()
        print(f"  [CREATED] account group '{name}' (id {r.json().get('id')})")
    except Exception as e:
        print(f"  [MANUAL]  could not verify/create account group '{name}': {e}")
        print("            -> create it in the Account console (User management -> Groups -> Add group)")


## Smoke Test (trainer, day before)

Run this the day **before** the training, after Steps 1–4. It iterates over every created per-user catalog and asserts that:

1. the catalog is reachable,
2. the `bronze` / `silver` / `gold` / `default` schemas exist,
3. the `datasets` Volume is accessible,
4. the key dataset files were copied (`customers.csv`, `orders_batch.json`, `products.csv`, `products.parquet`).

It prints **PASS / FAIL per user** and a final summary. Any FAIL must be fixed before Day 1 (re-run the affected step above for that user).


In [ ]:
# =============================================================================
# SMOKE TEST -- validate every participant environment (run the day before)
# =============================================================================
REQUIRED_SCHEMAS = [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA, DEFAULT_SCHEMA]
REQUIRED_FILES = [
    "customers/customers.csv",
    "orders/orders_batch.json",
    "products/products.csv",
    "products/products.parquet",
]

# Use the mapping from Step 1 if available; otherwise discover retailhub_* catalogs
try:
    catalogs_to_check = sorted(set(user_catalog_map.items()))
except NameError:
    discovered = [row[0] for row in spark.sql("SHOW CATALOGS").collect()
                  if row[0].startswith(f"{CATALOG_PREFIX}_")]
    catalogs_to_check = [("(discovered)", c) for c in sorted(discovered)]
    print(f"user_catalog_map not defined -- discovered {len(discovered)} '{CATALOG_PREFIX}_*' catalogs\n")

def smoke_test_catalog(catalog_name):
    """Return a list of failure messages (empty list = PASS)."""
    failures = []
    # 1. Catalog reachable
    try:
        spark.sql(f"USE CATALOG {catalog_name}")
    except Exception as e:
        return [f"catalog not reachable: {e}"]
    # 2. Schemas
    try:
        schemas = [row[0] for row in spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()]
        missing = [s for s in REQUIRED_SCHEMAS if s not in schemas]
        if missing:
            failures.append(f"missing schemas: {missing}")
    except Exception as e:
        failures.append(f"cannot list schemas: {e}")
    # 3. Volume + 4. dataset files
    volume_path = f"/Volumes/{catalog_name}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"
    try:
        dbutils.fs.ls(volume_path)
        for rel in REQUIRED_FILES:
            try:
                info = dbutils.fs.ls(f"{volume_path}/{rel}")
                if not info or info[0].size == 0:
                    failures.append(f"dataset file empty: {rel}")
            except Exception:
                failures.append(f"dataset file missing: {rel}")
    except Exception as e:
        failures.append(f"volume not accessible: {volume_path} ({e})")
    return failures

print("=" * 60)
print("SMOKE TEST -- per-user environments")
print("=" * 60)

results = {}
for email, catalog_name in catalogs_to_check:
    failures = smoke_test_catalog(catalog_name)
    results[catalog_name] = failures
    status = "PASS" if not failures else "FAIL"
    print(f"  [{status}] {catalog_name:<35} {email}")
    for f in failures:
        print(f"         - {f}")

print("=" * 60)
failed = {c: f for c, f in results.items() if f}
print(f"Smoke test: {len(results) - len(failed)}/{len(results)} environments PASS")
assert not failed, f"Smoke test FAILED for: {sorted(failed)} -- fix before Day 1"
print("ALL ENVIRONMENTS PASS -- ready for Day 1")

## Cleanup (After Training)

Run this section **after** the training to remove all participant catalogs and data.  
**WARNING: This is a destructive operation -- all training data will be permanently deleted!**

In [0]:
# =============================================================================
# CLEANUP -- Remove all training catalogs (after training)
# =============================================================================
# Uncomment the code below to execute cleanup.

# print("Dropping training catalogs...")
# for email, catalog_name in user_catalog_map.items():
#     try:
#         spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE")
#         print(f"  Dropped: {catalog_name}")
#     except Exception as e:
#         print(f"  Failed: {catalog_name} -- {e}")
# print("Cleanup complete.")

In [0]:
# =============================================================================
# ALTERNATIVE CLEANUP -- Find and remove all retailhub_* catalogs
# =============================================================================
# Use this if user_catalog_map is not available (e.g., new session).

# catalogs_df = spark.sql("SHOW CATALOGS")
# retailhub_catalogs = [row.catalog for row in catalogs_df.collect()
#                       if row.catalog.startswith("retailhub_")]

# print(f"Found {len(retailhub_catalogs)} catalogs to remove:")
# for cat in retailhub_catalogs:
#     print(f"  - {cat}")

# #Uncomment to drop:
# for cat in retailhub_catalogs:
#     spark.sql(f"DROP CATALOG IF EXISTS {cat} CASCADE")
#     print(f"Dropped: {cat}")